# Train Task 1 – MARIO Challenge

This notebook provides the training template for **Task 1** of the MICCAI 2024 MARIO Challenge.

It follows a simple workflow:
1. load the train and validation CSV files
2. build dataloaders
3. initialize the model
4. train and validate
5. save the best checkpoint

In this notebook, **OCTIP preprocessing was already applied (offline)** to accelerate training.
We used the train/validation split provided by the challenge organizers.

## 1. Imports and environment setup

In [2]:
import os
import random
import time
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import yaml

from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from pathlib import Path
# Add the project root to PYTHONPATH 
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from models.model import MarioModelT1
from utils.dataset import MARIO_DS_T1

# metrics
from utils.scoring import compute_metrics, specificity
import torch.nn.functional as F

# progress bar
from tqdm.notebook import tqdm

In [3]:
# Set a fixed random seed for reproducibility.
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [4]:
# Select the device used for training.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


## 2. Training configuration

In [5]:
# Define the training configuration used in this notebook.
TRAIN_CONFIG = {
    "train_csv_path": "../data/Task_1/df_task1_train.csv", # path to change 
    "val_csv_path": "../data/Task_1/df_task1_val.csv", # path to change 
    "train_data_root": "/media/pzhang/Lacie/pzhang/data/MARIO/Task_1/train/",  # path to change 
    "val_data_root": "/media/pzhang/Lacie/pzhang/data/MARIO/Task_1/val/",  # path to change 
    "image_size": (512, 200),
    "gray_scale": False,
    "train_batch_size": 24, # testing on local GPU 16GB
    "val_batch_size": 1,
    "num_workers": 4,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 2, # experiment
    "results_dir": "task1_output",
    "experiment_name": "resnet50_features_fusion",
    "backbone": "resnet50",
    "pretrained": True,
    "num_classes": 4,
    "model_type": "features_fusion",
}

## 3. Data loading

In [6]:
# Load the train and validation CSV files provided for Task 1.
df_train = pd.read_csv(TRAIN_CONFIG["train_csv_path"])[:5000] # subsampling for experiment
df_val = pd.read_csv(TRAIN_CONFIG["val_csv_path"])[:1000] # subsampling for experiment

print("Train size:", len(df_train))
print("Validation size:", len(df_val))

display(df_train.head())
display(df_val.head())

Train size: 5000
Validation size: 1000


,id_patient,side_eye,BScan,image_at_ti,image_at_ti+1,label,split_type,LOCALIZER_at_ti+1,LOCALIZER_at_ti,sex,age_at_ti+1,age_at_ti,num_current_visit_at_i+1,num_current_visit_at_i,delta_t,case
0,1,OD,2,E4521FF0.png,7AF1FF40.png,0,train,7A2E8830.png,E395D4D0.png,F,84,83,3,1,119,1
1,1,OD,3,E44AF400.png,7AE88960.png,0,train,7A2E8830.png,E395D4D0.png,F,84,83,3,1,119,2
2,1,OD,4,E4417E20.png,7ADEEC70.png,0,train,7A2E8830.png,E395D4D0.png,F,84,83,3,1,119,3
3,1,OD,5,E43A5230.png,7AD7E790.png,0,train,7A2E8830.png,E395D4D0.png,F,84,83,3,1,119,4
4,1,OD,6,E42E6B50.png,7ACE4AA0.png,0,train,7A2E8830.png,E395D4D0.png,F,84,83,3,1,119,5


,id_patient,side_eye,BScan,image_at_ti,image_at_ti+1,label,split_type,LOCALIZER_at_ti+1,LOCALIZER_at_ti,sex,age_at_ti+1,age_at_ti,num_current_visit_at_i+1,num_current_visit_at_i,delta_t,case
0,3,OS,2,B3944840.png,AA8E1A00.png,1,val,A9D689D0.png,B2EB0FF0.png,M,81,80,2,1,28,1170
1,3,OS,3,B38D4360.png,AA86EE10.png,1,val,A9D689D0.png,B2EB0FF0.png,M,81,80,2,1,28,1171
2,3,OS,4,B3813570.png,AA7D7830.png,1,val,A9D689D0.png,B2EB0FF0.png,M,81,80,2,1,28,1172
3,3,OS,5,B37A3090.png,AA73DB40.png,1,val,A9D689D0.png,B2EB0FF0.png,M,81,80,2,1,28,1173
4,3,OS,6,B37304A0.png,AA6A6560.png,1,val,A9D689D0.png,B2EB0FF0.png,M,81,80,2,1,28,1174


In [7]:
# Build the MARIO Task 1 datasets for training and validation.
train_set = MARIO_DS_T1(
    df_train,
    mode="train",
    image_size=TRAIN_CONFIG["image_size"],
    gray_scale=TRAIN_CONFIG["gray_scale"],
    root_dir=TRAIN_CONFIG["train_data_root"],
    processing_octip=False,
)

val_set = MARIO_DS_T1(
    df_val,
    mode="test",
    image_size=TRAIN_CONFIG["image_size"],
    gray_scale=TRAIN_CONFIG["gray_scale"],
    root_dir=TRAIN_CONFIG["val_data_root"],
    processing_octip=False,
)

print("Train dataset:", len(train_set))
print("Validation dataset:", len(val_set))

Train dataset: 5000
Validation dataset: 1000


In [8]:
# Build the dataloaders used during training and validation.
train_loader = DataLoader(
    train_set,
    batch_size=TRAIN_CONFIG["train_batch_size"],
    shuffle=True,
    num_workers=TRAIN_CONFIG["num_workers"],
    pin_memory=True,
)

val_loader = DataLoader(
    val_set,
    batch_size=TRAIN_CONFIG["val_batch_size"],
    shuffle=False,
    num_workers=TRAIN_CONFIG["num_workers"],
    pin_memory=True,
)

In [9]:
# Inspect one training batch to verify tensor shapes and labels.
images_t0, images_t1, labels, cases = next(iter(train_loader))

print("t0 batch shape:", images_t0.shape)
print("t1 batch shape:", images_t1.shape)
print("labels shape:", labels.shape)
print("cases example:", cases[:3])

t0 batch shape: torch.Size([24, 3, 200, 512])
t1 batch shape: torch.Size([24, 3, 200, 512])
labels shape: torch.Size([24])
cases example: tensor([1089, 2535, 2311])


In [10]:
# import matplotlib.pyplot as plt

# # Pick first sample of the batch
# img_t0 = images_t0[0].cpu()
# img_t1 = images_t1[0].cpu()

# # Convert from [C, H, W] → [H, W, C]
# img_t0 = img_t0.permute(1, 2, 0).numpy()
# img_t1 = img_t1.permute(1, 2, 0).numpy()

# # Plot
# fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# axes[0].imshow(img_t0, cmap="gray" if img_t0.shape[-1] == 1 else None)
# axes[0].set_title(f"T0 | label={labels[0].item()}")
# axes[0].axis("off")

# axes[1].imshow(img_t1, cmap="gray" if img_t1.shape[-1] == 1 else None)
# axes[1].set_title("T1")
# axes[1].axis("off")

# plt.tight_layout()
# plt.show()

## 4. Model, loss, and optimizer

In [11]:
# Initialize the Task 1 model and move it to the selected device.
model = MarioModelT1(
    TRAIN_CONFIG["backbone"],
    TRAIN_CONFIG["pretrained"],
    TRAIN_CONFIG["num_classes"],
    TRAIN_CONFIG["model_type"],
).to(DEVICE)

print(model.__class__.__name__)

MarioModelT1


In [12]:
# Define the loss function and optimizer used for training.
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)

## 5. Training and validation functions

In [13]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    running_samples = 0

    pbar = tqdm(loader, desc="Train", leave=False)

    for imgs_t0, imgs_t1, labels, _ in pbar:
        imgs_t0 = imgs_t0.to(device).float()
        imgs_t1 = imgs_t1.to(device).float()
        labels = labels.to(device).long()

        optimizer.zero_grad()

        logits = model(imgs_t0, imgs_t1)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs_t0.size(0)
        running_samples += imgs_t0.size(0)

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    epoch_loss = running_loss / running_samples
    return epoch_loss

In [14]:
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    running_samples = 0

    all_labels = []
    all_probs = []

    pbar = tqdm(loader, desc="Val", leave=False)

    for imgs_t0, imgs_t1, labels, _ in pbar:
        imgs_t0 = imgs_t0.to(device).float()
        imgs_t1 = imgs_t1.to(device).float()
        labels = labels.to(device).long()

        logits = model(imgs_t0, imgs_t1)
        loss = criterion(logits, labels)

        probs = F.softmax(logits, dim=1)

        running_loss += loss.item() * imgs_t0.size(0)
        running_samples += imgs_t0.size(0)

        all_labels.append(F.one_hot(labels, num_classes=4).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    epoch_loss = running_loss / running_samples

    y_true = np.concatenate(all_labels, axis=0)
    y_pred = np.concatenate(all_probs, axis=0)

    val_acc, _, _, val_f1, _, val_rkc = compute_metrics(y_true, y_pred)
    val_spec = specificity(np.argmax(y_true, axis=1), np.argmax(y_pred, axis=1))

    return epoch_loss, val_acc, val_f1, val_spec, val_rkc

## 6. Logging and checkpoint setup

In [15]:
# Create the output directory and TensorBoard writer used to store results.
results_path = os.path.join(TRAIN_CONFIG["results_dir"], TRAIN_CONFIG["experiment_name"])
os.makedirs(results_path, exist_ok=True)

best_model_path = os.path.join(results_path, "best_model_acc.pth")
writer = SummaryWriter(log_dir=results_path)

print("Results path:", results_path)

Results path: task1_output/resnet50_features_fusion


## 7. Training loop

In [16]:
# Run the training loop and save the best checkpoint according to validation accuracy.
best_val_acc = 0.0

for epoch in tqdm(range(TRAIN_CONFIG["num_epochs"]), desc="Epochs"):
    epoch_start = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, val_f1, val_spec, val_rkc = validate_one_epoch(model, val_loader, criterion, DEVICE)

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("ACC/val", val_acc, epoch)
    writer.add_scalar("F1/val", val_f1, epoch)
    writer.add_scalar("Specificity/val", val_spec, epoch)
    writer.add_scalar("RkC/val", val_rkc, epoch)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"[INFO] New best checkpoint saved at epoch {epoch} with val_acc={val_acc:.4f}")

    print(
        f"Epoch {epoch + 1}/{TRAIN_CONFIG['num_epochs']} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
        f"val_f1={val_f1:.4f} | val_spec={val_spec:.4f} | val_rkc={val_rkc:.4f} | "
        f"time={time.time() - epoch_start:.1f}s"
    )

writer.close()

Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

Train:   0%|          | 0/209 [00:00<?, ?it/s]

Val:   0%|          | 0/1000 [00:00<?, ?it/s]

[INFO] New best checkpoint saved at epoch 0 with val_acc=0.6090
Epoch 1/2 | train_loss=0.8441 | val_loss=0.9862 | val_acc=0.6090 | val_f1=0.4610 | val_spec=0.7500 | val_rkc=0.0000 | time=102.3s


Train:   0%|          | 0/209 [00:00<?, ?it/s]

Val:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 2/2 | train_loss=0.7106 | val_loss=1.0191 | val_acc=0.5930 | val_f1=0.4644 | val_spec=0.7471 | val_rkc=-0.0111 | time=102.2s


## 8. Reload the best checkpoint

In [19]:
# Reload the best model checkpoint saved during training.
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE, weights_only=True))
model.eval()

print("Best checkpoint loaded:", best_model_path)

Best checkpoint loaded: task1_output/resnet50_features_fusion/best_model_acc.pth
